# Antenna desig
---
Using the emerge library, we need to design an antenna. We chose to desing a yagi antenna, meaning it will be highly directional.

---
## Setup
First of all we need to install the emerge library. It only supports python between the version 3.10 and 3.13. Most of us will have the 3.14 version or higher installed. We need to create a virtual environment with python 3.12, source it and finally install the emerge package.
```
uv venv --python 3.12
source .venv/bin/activate
uv pip install emerge ipykernel
```

ALternatively, use ```uv pip install -r requirements.txt```, to install all dependencies

> [!WARNING]
> Use uv to create the venv **AND** to install via pip 

In [1]:
# === EMerge toolchain & Jupyter readiness check ===
# Run this cell first. Every line should read [OK]. Any [FAIL] tells you what to fix.
import sys
from importlib.metadata import version, PackageNotFoundError


def check(label, ok, detail=""):
    print(f"[{'OK  ' if ok else 'FAIL'}] {label}" + (f"  ->  {detail}" if detail else ""))
    return ok


print("=" * 62)
print(" EMerge toolchain / Jupyter readiness")
print("=" * 62)
ok_all = True

# 1) Python version -- EMerge requires >=3.10 and <3.14
v = sys.version_info
py_ok = (3, 10) <= (v.major, v.minor) <= (3, 13)
ok_all &= check(
    f"Python {v.major}.{v.minor}.{v.micro}",
    py_ok,
    "in 3.10-3.13" if py_ok else "EMerge needs 3.10-3.13 -> rebuild the venv with python 3.12",
)

# 2) Are we really inside the project venv (and not the system 3.14)?
in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix)
check("Virtual environment active", in_venv, sys.prefix if in_venv else "SYSTEM python, no venv")
print(f"        interpreter: {sys.executable}")

# 3) EMerge itself
try:
    ok_all &= check("EMerge installed", True, f"v{version('emerge')}")
except PackageNotFoundError:
    ok_all &= check("EMerge installed", False, "missing -> uv pip install emerge")
try:
    import emerge  # noqa: F401
    check("import emerge", True)
except Exception as e:
    ok_all &= check("import emerge", False, f"{type(e).__name__}: {e}")

# 4) Core dependencies: import AND report version
print("-" * 62)
for mod in ["numpy", "scipy", "matplotlib", "gmsh", "pyvista", "joblib", "loguru"]:
    try:
        __import__(mod)
        ok_all &= check(mod, True, f"v{version(mod)}")
    except Exception as e:
        ok_all &= check(mod, False, f"{type(e).__name__}: {e}")

# 5) numpy must satisfy EMerge's pin: >=2.2,<2.5 (the 3.14 trap you hit earlier)
try:
    import numpy as np
    nv = tuple(int(x) for x in np.__version__.split(".")[:2])
    ok_all &= check(f"numpy {np.__version__} matches EMerge pin (>=2.2,<2.5)", (2, 2) <= nv < (2, 5))
except Exception as e:
    check("numpy range check", False, str(e))

# 6) Jupyter / kernel readiness
print("-" * 62)
try:
    check("ipykernel installed (env selectable as a kernel)", True, f"v{version('ipykernel')}")
except PackageNotFoundError:
    ok_all &= check("ipykernel installed", False, "uv pip install ipykernel")
try:
    check("running inside a Jupyter/IPython kernel", True, type(get_ipython()).__name__)  # noqa: F821
except NameError:
    check("running inside a Jupyter/IPython kernel", False, "not a notebook kernel")

print("=" * 62)
print(" RESULT:", "toolchain READY" if ok_all else "NOT ready -> fix the [FAIL] lines above")
print("=" * 62)


2026-11-18 15:11:14.994 +01:00 | INFO     | emerge:<module>:60 - EMerge v3.0.9
2026-11-18 15:11:14.994 +01:00 | INFO     | emerge:<module>:61 - Importing modules


 EMerge toolchain / Jupyter readiness
[OK  ] Python 3.12.16  ->  in 3.10-3.13
[OK  ] Virtual environment active  ->  /home/spinaker/OneDrive_ecam/5Meo/High_Frequency/Project/Antenna/.venv
        interpreter: /home/spinaker/OneDrive_ecam/5Meo/High_Frequency/Project/Antenna/.venv/bin/python
[OK  ] EMerge installed  ->  v3.0.9
[OK  ] import emerge
--------------------------------------------------------------
[OK  ] numpy  ->  v2.2.6
[OK  ] scipy  ->  v1.18.1
[OK  ] matplotlib  ->  v3.11.2
[OK  ] gmsh  ->  v4.15.2
[OK  ] pyvista  ->  v0.49.2
[OK  ] joblib  ->  v1.6.0
[OK  ] loguru  ->  v0.8.0
[OK  ] numpy 2.2.6 matches EMerge pin (>=2.2,<2.5)
--------------------------------------------------------------
[OK  ] ipykernel installed (env selectable as a kernel)  ->  v7.4.0
[OK  ] running inside a Jupyter/IPython kernel  ->  ZMQInteractiveShell
 RESULT: toolchain READY


---
## Minimal EMerge test — inset-fed patch antenna

A trimmed version of the official `demo4_patch_antenna` to confirm the toolchain runs end-to-end: build geometry → mesh → solve a short frequency sweep → plot `S11`. The mesh is deliberately coarse and the interactive 3D viewer is removed, so this is a **smoke test, not an accurate design** (the resonance will land a bit off from the ~1.59 GHz target). A real FEM solve of the air sphere still takes a little time — let it run.

In [ ]:
# === Minimal EMerge test: inset-fed microstrip patch antenna (S11 only) ===
# Trimmed from the official examples/demo4_patch_antenna.py.
# Goal: confirm EMerge runs end-to-end (geometry -> mesh -> FEM solve -> S-params).
# The mesh is deliberately coarse and the 3D viewer calls are removed, so it is a
# SMOKE TEST, not an accurate design (expect the resonance a bit off from ~1.59 GHz).
import numpy as np
import emerge as em
from emerge.plot import plot_sp

mm = 0.001  # EMerge works in SI units

# --- Dimensions (inset-fed patch, resonance ~1.5-1.6 GHz) ---
Wpatch, Lpatch = 53 * mm, 52 * mm                  # patch
wline, wstub, lstub = 3.2 * mm, 7 * mm, 15.5 * mm  # feed line + inset notches
wsub, hsub, th = 100 * mm, 100 * mm, 1.524 * mm    # substrate (eps_r = 3.38)
Rair = 100 * mm                                    # radiation air sphere
f1, f2 = 1.545e9, 1.605e9                          # sweep band

model = em.Simulation("PatchAntennaTest")
model.check_version("3.0.9")

# --- Geometry ---
dielectric = em.geo.Box(wsub, hsub, th, position=(-wsub / 2, -hsub / 2, -th))
air = em.geo.Sphere(Rair).background()
ground = em.geo.XYPlate(wsub, hsub, position=(-wsub / 2, -hsub / 2, -th)).set_material(em.lib.PEC)

# Patch with inset feed: plate minus two notches, plus the feed line
rpatch = em.geo.XYPlate(Wpatch, Lpatch, position=(-Wpatch / 2, -Lpatch / 2, 0))
cutout1 = em.geo.XYPlate(wstub, lstub, position=(-wline / 2 - wstub, -Lpatch / 2, 0))
cutout2 = em.geo.XYPlate(wstub, lstub, position=(wline / 2, -Lpatch / 2, 0))
line = em.geo.XYPlate(wline, lstub, position=(-wline / 2, -Lpatch / 2, 0))
rpatch = em.geo.remove(rpatch, cutout1)
rpatch = em.geo.remove(rpatch, cutout2)
rpatch = em.geo.add(rpatch, line)
rpatch.set_material(em.lib.PEC)

# Lumped-port feed plate spanning the substrate thickness under the feed line
port = em.geo.Plate(
    np.array([-wline / 2, -Lpatch / 2, -th]),  # lower corner
    np.array([wline, 0, 0]),                   # width vector (x)
    np.array([0, 0, th]),                      # height vector (z)
)

dielectric.set_material(em.Material(3.38))

# --- Solver settings (coarse + short, for a quick test) ---
model.mw.set_resolution(0.2)             # fraction of wavelength (demo uses 0.15)
model.mw.set_frequency_range(f1, f2, 5)  # 5 points across the band

# --- Mesh ---
model.commit_geometry()
model.mesher.set_boundary_size(rpatch, 2 * mm)
model.mesher.set_face_size(port, 1 * mm)
model.generate_mesh()

# --- Boundary conditions: 50 ohm lumped port + absorbing outer sphere ---
model.mw.bc.LumpedPort(port, 1, width=wline, height=th, direction=em.ZAX, Z0=50)
model.mw.bc.AbsorbingBoundary(air.boundary())

# --- Solve ---
data = model.mw.run_sweep()

# --- Result: reflection coefficient S11 ---
fdense = np.linspace(f1, f2, 501)
S11 = data.scalar.grid.model_S(1, 1, fdense)
plot_sp(fdense, S11)  # return loss (dB) vs frequency

fres = fdense[np.argmin(np.abs(S11))]
print(f"[OK] EMerge solved the patch. Min |S11| ~ {20*np.log10(np.abs(S11).min()):.1f} dB "
      f"near {fres/1e9:.3f} GHz.")
